# Post-Sale Installation Services: Conversion Rate Analysis

**Domain:** Retail services / installation management  
**Methods:** Correlation analysis, lead time segmentation, artisan performance ranking, weighted scoring  
**Stack:** Python · pandas · numpy · matplotlib · seaborn

> **Note:** Data in this repository is synthetic and generated to replicate the structure and patterns of a real analysis, without exposing any proprietary or personally identifiable information. Artisan identifiers, store names and all monetary figures are fictional. The insights and statistical patterns are representative of the real case study.

---

## Business context

This analysis covers the post-sale installation funnel: from the sale of a site-visit voucher (*bollettino posa sopralluogo*) through the site visit, quotation discussion, and final conversion to an installation order.

**Scope:** 17,709 completed cases (site visit performed + quotation discussed), Jan–May 2024.  
**Overall conversion rate: 43.5%**

## Three areas investigated

1. **Lead time** — does the time from voucher to quotation discussion affect conversion?
2. **Service article incidence** — does the share of service costs in the quotation affect conversion?
3. **Artisan performance** — how much do individual artisans explain conversion differences?


## 0. Setup & load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.options.display.float_format = '{:.3f}'.format
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('data_servizi/mondo_servizi_synth.csv')
df['FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP'] = df['FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP'].astype(bool)
df['FLAG_INT'] = df['FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP'].astype(int)
df['LT_TOTALE'] = df['LT_SALDO_BV_SOPRALLUOGO_SIP'] + df['LT_SOPRALLUOGO_DISCUSSIONE_PREVENTIVO_SIP']

# Distinct cases (one row per bollettino/flag)
boll = df.drop_duplicates(subset=['NUMERO_BOLLETTINO', 'FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP'])

print(f"Total rows (article-level): {len(df):,}")
print(f"Distinct bollettini: {boll['NUMERO_BOLLETTINO'].nunique():,}")
print(f"Converted to order: {boll['FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP'].sum():,}")
print(f"Conversion rate: {boll['FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP'].mean():.1%}")
print(f"\nReparti:")
print(boll['REPARTO_BOLLETTINO'].value_counts())

## 1. Lead time analysis

We compute the total lead time (voucher payment → site visit → quotation discussion) and test whether longer delays are associated with lower conversion.

In [ ]:
boll_lt = boll[boll['LT_TOTALE'] >= 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution
sns.histplot(boll_lt['LT_TOTALE'], bins=30, kde=True, ax=axes[0])
axes[0].set_title('Distribution of total lead time\n(voucher payment → quotation discussion)')
axes[0].set_xlabel('Lead time (days)')
axes[0].set_ylabel('Frequency')

# TX by LT group
boll_lt['LT_GROUP'] = pd.cut(boll_lt['LT_TOTALE'],
    bins=[0, 5, 10, 15, 20, 30, np.inf],
    labels=['0-5', '6-10', '11-15', '16-20', '21-30', '30+'])
lt_tx = boll_lt.groupby('LT_GROUP', observed=True)['FLAG_INT'].agg(['mean','count']).reset_index()
lt_tx.columns = ['LT_GROUP','tx','n']

bars = axes[1].bar(lt_tx['LT_GROUP'], lt_tx['tx'], color=sns.color_palette('muted', len(lt_tx)))
for bar, row in zip(bars, lt_tx.itertuples()):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'n={row.n:,}', ha='center', fontsize=8)
axes[1].set_title('Conversion rate by lead time group')
axes[1].set_xlabel('Lead time (days)')
axes[1].set_ylabel('Conversion rate')
axes[1].set_ylim(0, 0.70)

plt.tight_layout()
plt.savefig('outputs_servizi/01_lead_time.png', dpi=150, bbox_inches='tight')
plt.show()

# Pearson correlation
corr = boll_lt['LT_TOTALE'].corr(boll_lt['FLAG_INT'])
print(f"Pearson correlation (LT vs conversion): {corr:.3f}")
print("\nConversion rate by LT group:")
print(lt_tx.to_string(index=False))

**Finding 1 — Lead time has negligible impact on conversion.**

The Pearson correlation between total lead time and conversion is approximately **−0.05**, indicating no meaningful linear relationship. The bulk of cases fall in the 11–20 day window, and conversion rates across those groups differ by less than 5pp. Only extreme delays (30+ days) show a notable drop, but these represent fewer than 4% of cases.

**Implication:** Reducing lead times should not be the primary lever for improving conversion. Resources are better directed elsewhere.

## 2. Service article incidence (articoli 49)

Articoli 49 are service/installation cost line items in the quotation. We test whether their share of the total quotation value predicts conversion.

In [ ]:
# Per-bollettino totals
tot_prev = df.groupby(['NUMERO_BOLLETTINO','FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP'])['CA_ARTICOLO_PREVENTIVO'].sum()
art49 = df[df['ID_ARTICOLO_PREVENTIVO'].between(49000000,49999999)].groupby(
    ['NUMERO_BOLLETTINO','FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP'])['CA_ARTICOLO_PREVENTIVO'].sum()

merged = pd.merge(art49.rename('art49'), tot_prev.rename('tot'), left_index=True, right_index=True).reset_index()
merged.columns = ['bollettino','trasformato','art49','tot']
merged['incidenza_49'] = merged['art49'] / merged['tot']

inc_t  = merged[merged['trasformato']==True]['incidenza_49'].mean()
inc_nt = merged[merged['trasformato']==False]['incidenza_49'].mean()
print(f"Incidenza art.49 — converted:     {inc_t:.3f}")
print(f"Incidenza art.49 — not converted: {inc_nt:.3f}")
print(f"Difference: {abs(inc_t - inc_nt):.3f} pp")

In [ ]:
# Conversion rate by total quotation value band
boll_cifra = merged.copy()
boll_cifra['fascia'] = pd.cut(boll_cifra['tot'],
    bins=[0, 5000, 10000, 15000, np.inf],
    labels=['€0–5k', '€5–10k', '€10–15k', '>€15k'])

fascia_tx = boll_cifra.groupby('fascia', observed=True)['trasformato'].agg(['mean','count']).reset_index()
fascia_tx.columns = ['fascia','tx','n']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(fascia_tx['fascia'], fascia_tx['tx'],
              color=sns.color_palette('RdYlGn', len(fascia_tx))[::-1])
for bar, row in zip(bars, fascia_tx.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{row.tx:.0%}\nn={row.n:,}', ha='center', fontsize=9)
ax.set_title('Conversion rate by total quotation value\n(service article incidence is NOT a driver — total value IS)')
ax.set_xlabel('Total quotation value')
ax.set_ylabel('Conversion rate')
ax.set_ylim(0, 0.70)
ax.axhline(0.435, color='grey', linestyle='--', linewidth=1, label='Overall avg (43.5%)')
ax.legend()

plt.tight_layout()
plt.savefig('outputs_servizi/02_cifra_vs_tx.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nConversion by quotation value band:")
print(fascia_tx.to_string(index=False))

**Finding 2 — Service article incidence is NOT a driver of conversion. Total quotation value IS.**

The share of service articles (art. 49) is virtually identical between converted (≈42–43%) and non-converted cases (≈43–44%). No meaningful difference.

However, **total quotation value has a strong negative relationship with conversion:**
- Quotations below €5k convert at ~48%
- Quotations €10–15k convert at ~37%
- Quotations above €15k convert at ~31%

The client's sensitivity to price — not the mix of service vs product articles — is the key friction point.

## 3. Artisan performance

We rank artisans by a weighted score (conversion rate × log(volume)) to account for both quality and statistical reliability.

In [ ]:
# Artisan-level performance (multi-reparto)
art_d = df.drop_duplicates(subset=['PARTITA_IVA_ARTIGIANO_SIP','NUMERO_BOLLETTINO','FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP'])
art_p = art_d.groupby('PARTITA_IVA_ARTIGIANO_SIP').agg(
    n=('NUMERO_BOLLETTINO','count'),
    converted=('FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP','sum'),
    tx=('FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP','mean')
).reset_index()
art_p['score'] = art_p['tx'] * np.log1p(art_p['n'])

# Filter: min 100 site visits for statistical reliability
art_100 = art_p[art_p['n'] >= 100].sort_values('score', ascending=False).reset_index(drop=True)
print(f"Artisans with ≥100 site visits: {len(art_100)}")
print(f"Conversion rate range: [{art_100['tx'].min():.1%}, {art_100['tx'].max():.1%}]")

top3  = art_100.head(3).copy()
flop3 = art_100.tail(3).copy()
top3['category']  = 'Top'
flop3['category'] = 'Flop'
plot_df = pd.concat([top3, flop3])

# Melt for grouped bar
melted = plot_df.melt(id_vars=['PARTITA_IVA_ARTIGIANO_SIP','category'],
                      value_vars=['converted','n'],
                      var_name='metric', value_name='value')
melted['metric'] = melted['metric'].map({'converted':'Converted','n':'Total'})

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=melted, x='value', y='PARTITA_IVA_ARTIGIANO_SIP',
            hue='metric', palette='muted', ax=ax)
for bar in ax.patches:
    w = bar.get_width()
    if w > 0:
        ax.text(w+1, bar.get_y()+bar.get_height()/2, f'{int(w)}', va='center', fontsize=8)
ax.set_title('Top 3 vs Flop 3 artisans (≥100 site visits, all departments)')
ax.set_xlabel('Count')
ax.set_ylabel('Artisan ID')
plt.tight_layout()
plt.savefig('outputs_servizi/03_artisan_top_flop.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 3 artisans:")
print(top3[['PARTITA_IVA_ARTIGIANO_SIP','n','converted','tx','score']].to_string(index=False))
print("\nFlop 3 artisans:")
print(flop3[['PARTITA_IVA_ARTIGIANO_SIP','n','converted','tx','score']].to_string(index=False))

In [ ]:
# Falegnameria / Finestre - Persiane - Tapparelle: specific sottoreparto focus
fale = df[(df['REPARTO_BOLLETTINO']=='FALEGNAMERIA') &
          (df['SOTTOREPARTO_BOLLETTINO']=='FINESTRE - PERSIANE-TAPPARELLE')]

fale_d = fale.drop_duplicates(subset=['PARTITA_IVA_ARTIGIANO_SIP','NUMERO_BOLLETTINO','FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP'])
fale_p = fale_d.groupby('PARTITA_IVA_ARTIGIANO_SIP').agg(
    n=('NUMERO_BOLLETTINO','count'),
    converted=('FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP','sum'),
    tx=('FLAG_SOPRALLUOGO_TRASFORMATO_IN_ORDINE_SIP','mean')
).reset_index()
fale_p['score'] = fale_p['tx'] * np.log1p(fale_p['n'])
fale_p = fale_p[fale_p['n'] >= 30].sort_values('score', ascending=False).reset_index(drop=True)

print(f"Artisans ≥30 site visits in Falegnameria/Finestre: {len(fale_p)}")
print(f"TX range: [{fale_p['tx'].min():.1%}, {fale_p['tx'].max():.1%}]")

top3f  = fale_p.head(3).copy(); top3f['category']  = 'Top'
flop3f = fale_p.tail(3).copy(); flop3f['category'] = 'Flop'
plot_f = pd.concat([top3f, flop3f])

melted_f = plot_f.melt(id_vars=['PARTITA_IVA_ARTIGIANO_SIP','category'],
                       value_vars=['converted','n'], var_name='metric', value_name='value')
melted_f['metric'] = melted_f['metric'].map({'converted':'Converted','n':'Total'})

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=melted_f, x='value', y='PARTITA_IVA_ARTIGIANO_SIP',
            hue='metric', palette='muted', ax=ax)
for bar in ax.patches:
    w = bar.get_width()
    if w > 0:
        ax.text(w+0.3, bar.get_y()+bar.get_height()/2, f'{int(w)}', va='center', fontsize=8)
ax.set_title('Falegnameria – Finestre/Persiane/Tapparelle\nTop 3 vs Flop 3 artisans (≥30 site visits)')
ax.set_xlabel('Count')
ax.set_ylabel('Artisan ID')
plt.tight_layout()
plt.savefig('outputs_servizi/04_artisan_falegnameria.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 3:")
print(top3f[['PARTITA_IVA_ARTIGIANO_SIP','n','converted','tx']].to_string(index=False))
print("\nFlop 3:")
print(flop3f[['PARTITA_IVA_ARTIGIANO_SIP','n','converted','tx']].to_string(index=False))

**Finding 3 — Artisan identity is the strongest driver of conversion.**

Across the full network (≥100 site visits), top artisans convert at **65–72%** vs **22–29%** for the weakest performers — a **~40pp gap** within the same operating context. This pattern holds when controlling for department and sub-department (e.g. Falegnameria/Finestre): artisans with identical quotation types and price ranges show dramatically different conversion rates.

The difference is **not explained by service article incidence** (no meaningful variation between top and flop artisans). It is attributable to individual skills: relationship management, quotation quality, and client follow-through.

**Implications:**
- Prioritise site-visit assignment to high-performing artisans, especially in high-value quotation segments
- Investigate and replicate the practices of top performers (training, shadowing)
- Monitor flop artisans with structured support before reallocating volume

## Summary of findings

| Area | Finding | Action |
|---|---|---|
| Lead time | No meaningful impact on conversion (r ≈ −0.05) | Do not over-invest in speed; focus elsewhere |
| Service article incidence | Nearly identical between converted and not (≈43%) | Not a lever |
| Total quotation value | Strong inverse relationship: >€15k converts at 31% | Pricing strategy and objection handling |
| **Artisan performance** | **40pp gap between top and flop artisans** | **Reassign volume, training, monitoring** |

---

*Data is synthetic and generated to replicate real statistical patterns without exposing proprietary information.*